# Practical 1: Feature Transformation using PCA
**Problem Statement:** Apply PCA on the Wine dataset (alcohol, ash, magnesium, etc.) so that most of the variance is captured by a small number of principal components, making it easier to distinguish the wine classes by inspecting those components.

**Dataset:** https://media.geeksforgeeks.org/wp-content/uploads/Wine.csv

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

%matplotlib inline

## 2. Load the Dataset
Tries the online link first. If there is no internet, place `Wine.csv` in the same folder as this notebook.

In [ ]:
url = "https://media.geeksforgeeks.org/wp-content/uploads/Wine.csv"
try:
    df = pd.read_csv(url)
except Exception:
    df = pd.read_csv("Wine.csv")

df.head()

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

## 3. Separate Features and Target
The last column holds the class label; all other columns are the measurements.

In [ ]:
target_col = df.columns[-1]
X = df.drop(columns=[target_col])
y = df[target_col]

print("Target column:", target_col)
print("Classes:", sorted(y.unique()))
print(y.value_counts())

## 4. Standardize the Features
PCA is sensitive to scale (e.g. Proline is in the hundreds, Hue is below 2), so every feature is scaled to mean 0 and variance 1.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pd.DataFrame(X_scaled, columns=X.columns).describe().loc[['mean', 'std']].round(2)

## 5. Apply PCA (all components) and check explained variance

In [ ]:
pca_full = PCA()
pca_full.fit(X_scaled)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

var_table = pd.DataFrame({
    "Component": [f"PC{i+1}" for i in range(len(explained))],
    "Explained Variance": explained.round(4),
    "Cumulative Variance": cumulative.round(4)
})
var_table

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

ax[0].bar(range(1, len(explained) + 1), explained, color="steelblue")
ax[0].set_title("Scree Plot (Explained Variance per Component)")
ax[0].set_xlabel("Principal Component")
ax[0].set_ylabel("Explained Variance Ratio")

ax[1].plot(range(1, len(cumulative) + 1), cumulative, marker="o", color="darkorange")
ax[1].axhline(0.95, color="red", linestyle="--", label="95% threshold")
ax[1].axhline(0.80, color="green", linestyle="--", label="80% threshold")
ax[1].set_title("Cumulative Explained Variance")
ax[1].set_xlabel("Number of Components")
ax[1].set_ylabel("Cumulative Variance")
ax[1].legend()

plt.tight_layout()
plt.show()

print("Components needed for 80% variance:", np.argmax(cumulative >= 0.80) + 1)
print("Components needed for 95% variance:", np.argmax(cumulative >= 0.95) + 1)

## 6. Reduce to 2 Principal Components

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=y, cmap='viridis')
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA of Wine Dataset")
plt.colorbar(label="Wine Class")
plt.show()
pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["Class"] = y.values

print("Original shape:", X_scaled.shape)
print("Reduced shape :", X_pca.shape)
print("Variance captured by PC1 + PC2: {:.2f}%".format(pca.explained_variance_ratio_.sum() * 100))
pca_df.head()

## 7. Visualize the Classes in PCA Space

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Class", palette="Set1", s=70, edgecolor="k")
plt.title("PCA of Wine Dataset (2 Components)")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.grid(alpha=0.3)
plt.show()

## 8. Which original features drive each component? (Loadings)

In [ ]:
loadings = pd.DataFrame(pca.components_.T, columns=["PC1", "PC2"], index=X.columns)
loadings.round(3)

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(loadings, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Feature Loadings on PC1 and PC2")
plt.show()

## 9. Verify: does PCA preserve the class information?
Train the same classifier on all original features and on only 2 principal components, then compare accuracy.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y)

# Model on all original features
clf_full = LogisticRegression(max_iter=1000)
clf_full.fit(X_train, y_train)
acc_full = accuracy_score(y_test, clf_full.predict(X_test))

# Model on 2 PCA components
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    X_pca, y, test_size=0.25, random_state=42, stratify=y)

clf_pca = LogisticRegression(max_iter=1000)
clf_pca.fit(Xp_train, yp_train)
y_pred_pca = clf_pca.predict(Xp_test)
acc_pca = accuracy_score(yp_test, y_pred_pca)

print(f"Accuracy with all {X.shape[1]} features : {acc_full:.4f}")
print(f"Accuracy with 2 principal components: {acc_pca:.4f}")

In [ ]:
cm = confusion_matrix(yp_test, y_pred_pca)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix (2 PCA Components)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 10. Conclusion
- The 13 original measurements were standardized and reduced with PCA.
- The first two principal components capture a little over half of the total variance (see the scree plot and variance table above), so 13 dimensions can be summarized by 2.
- The 2-D scatter plot shows the wine classes forming largely separate groups, so they can be told apart by inspecting PC1 and PC2.
- A classifier on only 2 components still performs well compared with one using all features, showing that PCA kept most of the information that matters while reducing dimensionality.